# HW0. Your first TPU run

This notebook has **no TPU attached**. That is deliberate.

Two people cannot share a Cloud TPU v5e chip. GKE schedules one pod per TPU node, and
that pod uses every chip on the node. If each of the 300 people in this course held a
chip for a full assignment, the course would need 300 chips. Most of that time nobody
would run code.

This notebook therefore runs on CPU. You send work to a chip when you have something to
run. `submit_tpu.run()` puts your code in a queue, waits for a free v5e chip, runs the
code there, and returns the output. You hold the chip only while your code runs.

The first run of the day takes two to three minutes. The cluster scales to zero, so GKE
must build a machine for you. Later runs take about 15 seconds, because a node from an
earlier job is usually still available.

In [ ]:
import submit_tpu

print(submit_tpu.run('''
import jax
print("devices:", jax.devices())
print("platform:", jax.devices()[0].platform)
'''))

You should see one `TpuDevice`. If you see a CPU device instead, something is wrong.
Tell a TA before you continue. Every number you measure after that point is a CPU
number with a TPU label on it.

## Part 1. Attention, and where the time goes

Below is a single multi-head attention forward pass, the shape that dominates an LLM
forward pass. Run it, then answer the questions underneath.

In [ ]:
print(submit_tpu.run('''
import time
import jax, jax.numpy as jnp

B, H, S, D = 8, 12, 1024, 64
key = jax.random.PRNGKey(0)
q, k, v = jax.random.normal(key, (3, B, H, S, D), dtype=jnp.bfloat16)

@jax.jit
def attn(q, k, v):
    logits = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(D).astype(q.dtype)
    return jnp.einsum("bhqk,bhkd->bhqd", jax.nn.softmax(logits, axis=-1), v)

attn(q, k, v).block_until_ready()          # compile first, time second
t0 = time.perf_counter()
for _ in range(50):
    out = attn(q, k, v)
out.block_until_ready()
dt = (time.perf_counter() - t0) / 50

flops = 4 * B * H * S * S * D
print(f"{dt*1e3:.3f} ms per pass")
print(f"{flops/dt/1e12:.2f} TFLOP/s")
'''))

**Q1.** A v5e chip peaks near 197 TFLOP/s in bfloat16. You measured far less than that.
Work out how many bytes the `logits` array occupies and how many bytes the chip must
move to compute the softmax over it. Is this kernel limited by arithmetic or by memory
bandwidth? Show the arithmetic intensity you calculated.

**Q2.** Double `S` to 2048 and run again. Predict the change in time *before* you run
it, then explain any gap between your prediction and the measurement.

## Part 2. Make it faster

The two-einsum version above materialises the whole `B × H × S × S` logits array in
memory. Flash attention avoids that by tiling over the sequence and keeping a running
softmax, so the large intermediate never exists.

Use `jax.nn.dot_product_attention` and compare. Report both numbers and explain the
difference in terms of the bytes moved, not just the wall time.

In [ ]:
# Your code here. Keep it inside a submit_tpu.run('''...''') block. The notebook
# itself has no chip.


## Checking your place in the queue

If a run is slow, it is either waiting for a chip or waiting for a machine. These tell
you which:

```bash
kubectl get workloads          # ADMITTED False means you are queued behind others
kubectl get localqueue tpu     # how many are pending in your section
```

`ADMITTED True` with the pod still `Pending` means a chip is reserved for you and a
node is being built. That is normal and takes a couple of minutes.

## Submitting

Hand in this notebook with all outputs kept. The output of a `submit_tpu.run()` call
carries the device it ran on, so we can see the work happened on a real chip.